# Lekcia 10 - AI agenti v produkcii

V tejto lekcii sa naučíte **produkčné vzory** pre AI agentov používajúce Microsoft Agent Framework (Python). Pokrývame:

- **Pozorovateľnosť** — pridávanie merania času a protokolovania do interakcií agenta
- **Hodnotenie** — použitie hodnotiaceho agenta na ohodnotenie kvality odpovedí
- **Riadenie nákladov** — stratégie pre optimalizáciu tokenov a výber modelu

Scenár predstavuje **cestovného agenta**, ktorý pomáha používateľom plánovať cesty, pričom nad tým sú vrstvené monitorovanie a hodnotenie.


## Nastavenie


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Prevádzkové úvahy

Presun AI agentov z prototypov do produkcie vyžaduje dôkladnú pozornosť trom pilierom:

1. **Sledovateľnosť** — Potrebujete prehľad o tom, čo agent robí, ako dlho to trvá, a ktoré nástroje volá. Bez trasovania a logovania je ladenie problémov v produkcii takmer nemožné.

2. **Hodnotenie** — Automatizované kontroly kvality zabezpečujú, že odpovede agenta zostávajú presné, úplné a užitočné v priebehu času. Hodnotiaci agent môže ohodnotiť odpovede podľa definovaných kritérií.

3. **Riadenie nákladov** — Spotreba tokenov priamo ovplyvňuje náklady. Stratégie ako optimalizácia promptov, výber modelu a kešovanie pomáhajú udržiavať výdavky pod kontrolou bez obetovania kvality.


## Vytváranie pozorovateľného agenta

Definujeme nástroje na cestovanie a obalíme volanie agenta časovaním, aby sme mohli sledovať latenciu. V produkcii by ste integrovali OpenTelemetry alebo podobný backend na trasovanie.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vzory hodnotenia

Bežným produkčným vzorom je použiť druhého agenta ako **hodnotiteľa**. Hodnotiteľ priraďuje skóre odpovedi primárneho agenta na základe preddefinovaných kritérií, ako sú úplnosť, presnosť a užitočnosť.

To umožňuje:
- Automatizované brány kvality pred doručením odpovedí používateľom
- Detekcia regresií pri zmene promptov alebo modelov
- Neustále monitorovanie výkonu agenta v priebehu času


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Stratégie riadenia nákladov

Kontrola nákladov je kľúčová pre produkčných AI agentov. Tu sú hlavné stratégie:

| Strategy | Description |
|---|---|
| **Optimalizácia promptu** | Udržujte systémové inštrukcie stručné. Odstráňte redundantný kontext, aby ste znížili počet vstupných tokenov. |
| **Výber modelu** | Používajte menšie, lacnejšie modely (napr. GPT-4o-mini) pre jednoduché úlohy ako klasifikácia alebo extrakcia a väčšie modely si nechajte pre zložité uvažovanie. |
| **Kešovanie** | Kešujte výsledky nástrojov a časté dopyty, aby ste sa vyhli zbytočným volaniam API. |
| **Rozpočty tokenov** | Nastavte limity `max_tokens`, aby ste predišli neočakávane dlhým odpovediam. |
| **Spracovanie dávkami** | Zoskupte viacero používateľských dopytov do jedného volania API, kde je to možné. |

V praxi funguje dobre viacvrstvový prístup: nasmerujte jednoduché požiadavky na rýchly, lacný model a len zložité dopyty eskalujte na výkonnejší model.


## Zhrnutie

V tejto lekcii ste sa naučili, ako:

1. **Pridať pozorovateľnosť** do interakcií agenta s meraním času a logovaním, čím sa položia základy pre trasovanie a monitorovanie.
2. **Hodnotiť odpovede agenta** automaticky pomocou hodnotiaceho agenta, ktorý priraďuje skóre úplnosti, presnosti a užitočnosti.
3. **Riadiť náklady** prostredníctvom optimalizácie promptov, výberu modelu, kešovania a rozpočtov tokenov.

Tieto produkčné vzory pomáhajú zabezpečiť, že vaše AI agenti sú spoľahliví, merateľní a nákladovo efektívni pri škálovaní.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vylúčenie zodpovednosti:
Tento dokument bol preložený pomocou služby automatického prekladu založenej na umelej inteligencii Co-op Translator (https://github.com/Azure/co-op-translator). Hoci sa usilujeme o presnosť, berte prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho pôvodnom jazyku by sa mal považovať za rozhodujúci zdroj. Pre dôležité informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
